# 1. Arquitectura y Fundamentos de Modelos NMS-Free

En la detección de objetos convencional, la etapa de inferencia dependía críticamente de la supresión de no máximos o **NMS (Non-Maximum Suppression)**. Este algoritmo heurístico de postprocesamiento opera filtrando de forma iterativa todas las cajas de predicción que superen un umbral de solapamiento predefinido (Intersection over Union, $\text{IoU}$). El proceso introduce un cuello de botella computacional masivo y no determinista, cuya latencia escala exponencialmente con la densidad de objetos en la escena, impidiendo un rendimiento óptimo en arquitecturas de hardware de cómputo embebido o de borde.

Los paradigmas modernos resuelven este problema redefiniendo la estrategia de asignación de etiquetas (*label assignment*) directamente durante el entrenamiento, eliminando el NMS por completo de la ecuación de inferencia a través de dos vertientes distintas: la optimización de emparejamiento uno-a-uno convolucional (YOLOv26) y la optimización basada en la teoría de grafos y atención global (RT-DETR).

---

## 1.1. YOLOv26: Evolución End-to-End en Modelos Convolucionales / CSP

YOLOv26 hereda la eficiencia estructural de las redes convolucionales basadas en bloques CSP (Cross Stage Partial), pero elimina las dos grandes fuentes de latencia computacional que afectaban a sus predecesores: el postprocesamiento **NMS** y el cálculo de la pérdida de focalización distributiva o **DFL (Distribution Focal Loss)**. 

### 1.1.1. Backbone (Extracción Jerárquica de Características)
El flujo de datos inicia con la inyección de un tensor de imagen de entrada $I \in \mathbb{R}^{B \times 3 \times H \times W}$, donde $B$ es el tamaño de lote (*batch size*), $3$ representa los canales RGB, y $H, W$ denotan las dimensiones espaciales iniciales. 

A través de una sucesión jerárquica de bloques convolucionales avanzados y módulos CSP modificados que reducen la resolución espacial mediante convoluciones con zancada (*stride*), la red extrae mapas de características multiescala. Las salidas críticas del Backbone se extraen en tres niveles específicos de resolución espacial, denominados formalmente:
*   **P3 (Especializado en objetos pequeños):** Mantiene una resolución espacial alta gracias a un factor de submuestreo de $8\times$. Su forma matemática está dada por el tensor:
    $$\text{Shape}_{P3} = \left( B, C_3, \frac{H}{8}, \frac{W}{8} \right)$$
*   **P4 (Especializado en objetos medianos):** Submuestreo intermedio de $16\times$, balanceando resolución espacial e información semántica abstracta:
    $$\text{Shape}_{P4} = \left( B, C_4, \frac{H}{16}, \frac{W}{16} \right)$$
*   **P5 (Especializado en objetos grandes):** Submuestreo máximo de $32\times$, capturando campos receptivos amplios con alta densidad semántica:
    $$\text{Shape}_{P5} = \left( B, C_5, \frac{H}{32}, \frac{W}{32} \right)$$

### 1.1.2. Neck (Fusión de Características e Invariabilidad de Red)
Las características crudas extraídas $\{P3, P4, P5\}$ ingresan a una red de agregación de caminos bidireccional basada en la arquitectura PANet (*Path Aggregation Network*). El Neck combina las características semánticas de alto nivel con los detalles geométricos de bajo nivel mediante dos flujos simultáneos:
1.  **Flujo Top-Down:** Toma el mapa más profundo (P5), reduce sus canales mediante convoluciones $1\times1$, aplica una interpolación espacial por vecinos más cercanos (*Upsampling*) con un factor de $2\times$, aumentando su tamaño espacial para acoplarse y concatenarse directamente sobre el canal del tensor P4. Este proceso se repite recursivamente de P4 hacia P3.
2.  **Flujo Bottom-Up:** Toma las características refinadas de P3, reduce sus dimensiones espaciales mediante convoluciones $3\times3$ con zancada $2$ (*Downsampling*), concatenando el resultado de vuelta en los niveles superiores P4 y P5.

Durante todo este proceso de agregación y combinación de canales, las dimensiones espaciales se mantienen estrictamente invariantes respecto a sus escalas originales $\frac{H}{8}, \frac{H}{16}, \frac{H}{32}$. Lo que se altera de forma dinámica es la profundidad de los canales ($C'_3, C'_4, C'_5$), que se homogeneizan y enriquecen semánticamente mediante los bloques de convolución internos del Neck.

### 1.1.3. Head (Mecanismo Dual y Predicción Uno-a-Uno Sin DFL)
El avance fundamental de YOLOv26 se encuentra en su **Cabeza de Detección Dual (Dual-Head)**, diseñada estratégicamente para desacoplar el comportamiento del modelo durante las fases de entrenamiento e inferencia.

*   **Durante el Entrenamiento:** Los mapas de características del Neck alimentan en paralelo a dos cabezas predictoras independientes: una **Cabeza Uno-a-Muchos (1-to-Many Head)** y una **Cabeza Uno-a-Uno (1-to-1 Head)**. La cabeza uno-a-muchos utiliza asignadores dinámicos tradicionales (como TAL - Task Alignment Learning), donde múltiples anclajes o celdas espaciales se asignan a una misma etiqueta real (*ground truth*). Esto maximiza la riqueza de las señales de gradiente y acelera la optimización del Backbone. 
*   **Durante la Inferencia:** La cabeza uno-a-muchos se descarta por completo. El modelo se ejecuta de extremo a extremo utilizando exclusivamente la **Cabeza Uno-a-Uno (1-to-1 Head)**. Esta cabeza implementa un algoritmo de asignación estricta que penaliza las predicciones duplicadas para un mismo objeto. Forzado por funciones de costo de emparejamiento por proximidad, el modelo aprende a emitir una única caja de alta confianza por cada objeto físico real presente en la escena.

*   **Eliminación de la Pérdida de Focalización Distributiva (DFL):** Las versiones anteriores de YOLO dividían el espacio continuo de las coordenadas de las cajas de delimitación en una distribución de probabilidad discreta de $16$ bins para manejar la incertidumbre de los bordes difusos. Esto requería que el Head proyectara un vector de tamaño $4 \times 16 = 64$ canales para la regresión de la caja. 
    YOLOv26 elimina esta complejidad matemática, volviendo a una regresión directa y lineal de los deltas de las coordenadas $[x, y, w, h]$. Esto reduce drásticamente los canales de salida del Head y el costo computacional asociado.

Al unificar las salidas espaciales de los tres niveles mediante un aplanamiento y filtrado interno de las propuestas con mayor confianza estadística, el tensor final de salida colapsa a una matriz bidimensional estática en inferencia:
$$\text{Shape de Salida (Inferencia YOLOv26)} = \left( B, N_{\text{max}}, 4 + N_{\text{clases}} \right)$$
Donde $N_{\text{max}}$ es un hiperparámetro fijo por diseño (por defecto $300$). Como cada uno de los $N_{\text{max}}$ vectores contiene directamente las coordenadas unívocas de la caja y sus respectivas probabilidades de clase sin duplicados espaciales, el hardware no ejecuta operaciones adicionales de ordenamiento por IoU, eliminando el NMS.

---

## 1.2. RT-DETR: El Enfoque Basado en Transformers en Tiempo Real

RT-DETR rompe con el paradigma convolucional puro al trasladar el problema de la detección de objetos hacia una formulación de optimización global basada en secuencias de tokens e interacciones a largo alcance, resolviendo la altísima latencia que penalizaba a los DETR (*DEtection Transformers*) tradicionales.

### 1.2.1. Backbone
Al igual que en los modelos YOLO, la imagen $I \in \mathbb{R}^{B \times 3 \times H \times W}$ se procesa mediante un extractor base altamente optimizado para operaciones en paralelo (típicamente arquitecturas como ResNet o HGNetv2). El resultado es la obtención de mapas de características multiescala estructurados jerárquicamente en los niveles estándar de resolución:
$$\mathcal{F} = \{S_3, S_4, S_5\}$$
Cuyas dimensiones corresponden exactamente a escalas espaciales de $\frac{H}{8}, \frac{H}{16}, \frac{H}{32}$ respecto al espacio nativo de la imagen.

### 1.2.2. Neck (Efficient Hybrid Encoder y la Transformación del Shape)
Los modelos DETR clásicos introducían una latencia inasumible en tiempo real debido a que aplicaban capas de auto-atención global (*Self-Attention*) multiescala sobre la totalidad de los píxeles de todos los mapas de características. Dado que la complejidad computacional de la auto-atención escala cuadráticamente ($O(S^2)$), calcular la atención sobre resoluciones altas como $S_3$ saturaba la memoria.

RT-DETR soluciona este cuello de botella introduciendo el **Codificador Híbrido Eficiente (Efficient Hybrid Encoder)**, el cual desacopla la extracción en dos fases matemáticas y de forma secuencial:
1.  **AIE (Attention-based Intra-scale Feature Interaction):** El modelo reconoce que la información de largo alcance y el contexto global son críticos principalmente en el mapa de características con mayor abstracción semántica ($S_5$). Por lo tanto, extrae únicamente $S_5$, lo aplana espacialmente y le aplica capas de auto-atención basadas en Transformers. Las escalas de menor abstracción y mayor resolución ($S_3$ y $S_4$) eluden este paso de alta densidad de cómputo.
2.  **CCFI (CNN-based Cross-scale Feature Fusion):** Utiliza bloques basados exclusivamente en convoluciones nativas de bajo costo para fusionar jerárquicamente el mapa enriquecido globalmente con las características de las escalas inferiores ($S_4$ y $S_3$) mediante conexiones *Top-Down* y *Bottom-Up*.

*   **Modificación Matemática del Shape:** Tras la fusión e interacción, el codificador híbrido proyecta todos los mapas a una dimensión de embedding común y homogénea denotada como $d$ (por ejemplo, $d = 256$). Posteriormente, realiza una operación de aplanamiento de las dimensiones espaciales de cada mapa de características ($H_i \times W_i$) y los concatena a lo largo del eje secuencial. 
    El tensor resultante pasa de una representación espacial 2D a una secuencia pura de tokens lista para el Transformer:
    $$\text{Shape de Salida del Encoder} = \left( B, L, d \right) \quad \text{donde} \quad L = \sum_{i=3}^{5} (H_i \times W_i)$$

### 1.2.3. Decoder y Head (Query-Based Prediction y la Asignación Húngara)
El Decodificador del Transformer toma la secuencia de tokens del codificador $\left( B, L, d \right)$ y un conjunto estático de **Consultas de Objetos (Object Queries)** inicializadas mediante una selección previa basada en las puntuaciones de las características del codificador (*Query Selection*). El número de consultas se limita estrictamente a un tamaño fijo $Q$ (normalmente $Q = 300$).

Cada consulta es un vector continuo de dimensión $d$ que actúa como una ranura de búsqueda (*slot*) en la escena. A través de capas alternas de **Auto-Atención** (donde las consultas interactúan entre sí para coordinar qué objeto buscará cada una) y **Atención Cruzada (Cross-Attention)** (donde las consultas extraen información del mapa de características global $L$), las consultas se refinan capa por capa.

*   **Mecanismo NMS-Free (Asignación de Coincidencia Húngara):** El motivo por el cual RT-DETR elimina por completo la necesidad de un postprocesamiento NMS radica en su formulación de pérdida basada en la teoría de grafos. Durante el entrenamiento, se realiza un **emparejamiento bipartito único** entre el conjunto cerrado de las $Q$ predicciones del modelo y el conjunto de etiquetas reales anotadas en la imagen (completado con elementos vacíos $\emptyset$ si el número de objetos reales es menor que $Q$).
    Este emparejamiento se resuelve de manera exacta mediante el **Algoritmo Húngaro**, el cual busca la permutación óptima de asignación $\sigma$ dentro del espacio de permutaciones posibles $\mathfrak{S}_Q$ que minimice un costo de coincidencia global:
    $$\hat{\sigma} = \arg\min_{\sigma \in \mathfrak{S}_Q} \sum_{i=1}^{Q} \mathcal{L}_{\text{match}}(y_i, \hat{y}_{\sigma(i)})$$
    La función de costo $\mathcal{L}_{\text{match}}$ penaliza simultáneamente los errores de clasificación y las discrepancias de localización espacial (mediante distancias $\ell_1$ e $\text{GIoU}$). 
    
    Al forzar este emparejamiento uno-a-uno matemático estricto durante la optimización del gradiente, y gracias a que la capa de auto-atención permite a las consultas comunicarse explícitamente entre sí para evitar solaparse ("si la consulta A ya se enfocó en esta semilla, la consulta B se enfoca en otra o se apaga"), el decodificador genera de forma natural predicciones disjuntas. No existen múltiples cajas compitiendo por un mismo objeto físico.

Finalmente, los embeddings de salida de las consultas refinadas se proyectan de forma paralela mediante redes lineales totalmente conectadas (MLP - Multi-Layer Perceptrons) para decodificar las coordenadas espaciales y las probabilidades de pertenencia a clase:
$$\text{Shape Final de Salida (Inferencia RT-DETR)} = \left( B, Q, 4 + N_{\text{clases}} \right)$$
Cada uno de los $Q$ elementos entrega su resultado de manera independiente y directa. Si una consulta no detectó ningún objeto, su probabilidad de clase se concentrará en la categoría de fondo, permitiendo un filtrado por umbral de confianza plano sin recurrir a cálculos geométricos de IoU cruzados o NMS.
```

# 2. Configuración e Implementación Práctica con Ultralytics

En esta sección prepararemos el entorno para entrenar y evaluar modelos libres de NMS en una tarea de detección de semillas utilizando `ultralytics`.

In [ ]:
import os
import yaml
import shutil
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from roboflow import Roboflow
from ultralytics import YOLO, RTDETR

# Desactivar logging de Weights & Biases para acelerar el inicio
os.environ['WANDB_DISABLED'] = 'true'

## 2.1. Configuración de Parámetros de la Sesión

Definimos las variables de configuración global para el dataset, los hiperparámetros de entrenamiento y la disponibilidad de hardware para los modelos convolucional y Transformer.

In [ ]:
# Parámetros de Roboflow
ROBOFLOW_API_KEY = "3ioUIbqERJ2jEWElELQN"
PROJECT_NAME = "seed-detection-smrzf"
DATASET_VERSION = 6

# Hiperparámetros de Entrenamiento y Validación
EPOCHS = 30
IMAGE_SIZE = 128
BATCH_SIZE = 16
CONF_THRESHOLD = 0.8

# Detección y configuración de hardware de GPU / CPU
# csv_devices_exist se establece en True si hay más de 1 GPU (para usar device=[0,1])
csv_devices_exist = torch.cuda.is_available() and torch.cuda.device_count() > 1
device_train = [0, 1] if csv_devices_exist else ('0' if torch.cuda.is_available() else 'cpu')
device_val = '0' if torch.cuda.is_available() else 'cpu'

print(f"GPUs múltiples detectadas (csv_devices_exist): {csv_devices_exist}")
print(f"Dispositivo de entrenamiento asignado: {device_train}")
print(f"Dispositivo de validación asignado: {device_val}")

## 2.2. Descarga del Dataset vía Roboflow

Descargamos las imágenes etiquetadas para la detección de semillas germinadas y no germinadas en formato YOLOv11.

In [ ]:
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("gcpds-tm2ae").project(PROJECT_NAME)
version = project.version(DATASET_VERSION)
dataset = version.download("yolov11")

# Reorganizar directorios según el flujo estándar de Ultralytics
if not os.path.exists('./datasets'):
    os.makedirs('./datasets')
if os.path.exists('./Seed-Detection-6') and not os.path.exists('./datasets/Seed-Detection-6'):
    shutil.move('./Seed-Detection-6', './datasets/')

## 2.3. Configuración y Corrección del archivo YAML

Reescribimos el archivo de configuración `data.yaml` para asegurar que las rutas locales de entrenamiento, validación y prueba estén alineadas con el directorio `./datasets`.

In [ ]:
data_yaml_config = {
    'path': './Seed-Detection-6',
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': {0: 'Germinada', 1: 'No germinada'},
    'roboflow': {
        'license': 'CC BY 4.0',
        'project': PROJECT_NAME,
        'url': f'https://universe.roboflow.com/gcpds-tm2ae/{PROJECT_NAME}/dataset/{DATASET_VERSION}',
        'version': DATASET_VERSION,
        'workspace': 'gcpds-tm2ae'
    }
}

file_path = './datasets/Seed-Detection-6/data.yaml'
with open(file_path, 'w') as yaml_file:
    yaml.dump(data_yaml_config, yaml_file, default_flow_style=False)

## 2.4. Entrenamiento y Validación con YOLOv26 (NMS-Free CNN)

Cargamos el modelo preentrenado compacto `yolo26s.pt` e iniciamos el entrenamiento y la validación utilizando asignación directa uno-a-uno.

In [ ]:
print("--- Iniciando entrenamiento con YOLOv26 ---")
# Carga del modelo preentrenado compacto de la familia YOLOv26
model_yolo26 = YOLO('yolo26s.pt')

# Entrenamiento
model_yolo26.train(
    data=file_path,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    device=device_train
)

# Limpieza previa de validaciones antiguas para evitar colisiones de rutas en kaggle o local
for val_path in ['./runs/detect/val', '/kaggle/working/runs/detect/val']:
    if os.path.exists(val_path):
        shutil.rmtree(val_path)

# Validación de YOLOv26
# Nota de cátedra: Al ser arquitecturas NMS-Free nativas o configuradas de extremo a extremo,
# el parámetro 'iou' pierde la relevancia crítica de filtrado heurístico tradicional en inferencia.
val_results_yolo26 = model_yolo26.val(
    data=file_path,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    conf=CONF_THRESHOLD,
    device=device_val
)

# Visualización de métricas de YOLOv26
fig, axis = plt.subplots(1, 2, figsize=(18, 8))
if os.path.exists("./runs/detect/val/confusion_matrix_normalized.png"):
    img_cm = mpimg.imread("./runs/detect/val/confusion_matrix_normalized.png")
    img_f1 = mpimg.imread("./runs/detect/val/BoxF1_curve.png")
    axis[0].imshow(img_cm)
    axis[0].set_title("YOLOv26: Matriz de Confusión")
    axis[1].imshow(img_f1)
    axis[1].set_title("YOLOv26: Curva F1-Score")
else:
    print("Los gráficos de validación se almacenaron en la ruta de ejecución actual.")
[ax.axis('off') for ax in axis]
plt.show()

## 2.5. Entrenamiento y Validación con RT-DETR (NMS-Free Transformer)

Procedemos a cargar el detector basado en consultas y decodificadores Transformer en tiempo real (`rtdetr-l.pt`), entrenándolo y evaluándolo bajo el mismo escenario de prueba.

In [ ]:
print("--- Iniciando entrenamiento con RT-DETR ---")
# Carga del modelo base de RT-DETR mediante el submódulo específico de Ultralytics
model_rtdetr = RTDETR('rtdetr-l.pt')

# Entrenamiento
model_rtdetr.train(
    data=file_path,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    device=device_train
)

# Para RT-DETR, limpiamos runs/detect/val2 si existiera para que los resultados queden en una ruta limpia
for val_path in ['./runs/detect/val2', '/kaggle/working/runs/detect/val2']:
    if os.path.exists(val_path):
        shutil.rmtree(val_path)

# Validación de RT-DETR
# Observar que el procesamiento no requiere sintonizar umbrales NMS complejos.
val_results_rtdetr = model_rtdetr.val(
    data=file_path,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    conf=CONF_THRESHOLD,
    device=device_val
)

# Visualización de métricas de RT-DETR
fig, axis = plt.subplots(1, 2, figsize=(18, 8))

# El directorio de validación puede ser runs/detect/val2 debido al guardado secuencial de runs
val_dir = "./runs/detect/val"
if os.path.exists("./runs/detect/val2"):
    val_dir = "./runs/detect/val2"
elif os.path.exists("/kaggle/working/runs/detect/val2"):
    val_dir = "/kaggle/working/runs/detect/val2"
elif os.path.exists("/kaggle/working/runs/detect/val"):
    val_dir = "/kaggle/working/runs/detect/val"

if os.path.exists(f"{val_dir}/confusion_matrix_normalized.png"):
    img_cm = mpimg.imread(f"{val_dir}/confusion_matrix_normalized.png")
    img_f1 = mpimg.imread(f"{val_dir}/BoxF1_curve.png")
    axis[0].imshow(img_cm)
    axis[0].set_title("RT-DETR: Matriz de Confusión")
    axis[1].imshow(img_f1)
    axis[1].set_title("RT-DETR: Curva F1-Score")
else:
    print(f"Los gráficos de validación se almacenaron en la ruta: {val_dir}")
[ax.axis('off') for ax in axis]
plt.show()